# 🎓 Módulo 06: Post-Training, Fine-Tuning y el Arte del LLM
## Capítulo 1: Supervised Fine-Tuning (SFT), Plantillas de Chat (ChatML) y Enmascaramiento de Pérdida

> *"Un modelo de lenguaje base pre-entrenado (como GPT-3 o LLaMA base) no es un asistente: si le preguntas '¿Cuál es la capital de Francia?', no te responderá 'La capital es París', sino que podría autocompletar con '¿Cuál es la capital de Alemania? ¿Cuál es la capital de Italia?'. El pre-entrenamiento acumula conocimiento del mundo comprimiendo internet; el post-entrenamiento (SFT) enseña modales, formato y obediencia a instrucciones. El truco algorítmico maestro consiste en enmascarar los tokens del prompt con un índice de ignorado (-100) para que el modelo solo aprenda a predecir las respuestas del asistente."*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mcarbonell/algo-to-ai/blob/main/notebooks/06_post_training/01_supervised_fine_tuning_sft.ipynb)

---

### ⚙️ Inicialización del Entorno
Cargamos las librerías matemáticas y fijamos semillas para asegurar reproducibilidad determinista.

In [ ]:
# !pip install -q numpy matplotlib torch
from typing import Tuple, List, Dict, Optional, Any
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)
print("✅ Entorno listo para estudiar Supervised Fine-Tuning (SFT) from scratch")

---

## 1. 📜 Contexto Histórico y Proceso de Descubrimiento

### El Paradigma del Pre-entrenamiento (2018 - 2020)
Con modelos como GPT-2 y GPT-3 (Brown et al., 2020), la industria descubrió que pre-entrenar Transformers gigantes sobre terabytes de texto web permite aprender gramática, hechos históricos, sintaxis de programación y capacidades emergentes.

Sin embargo, **un modelo base es solo un completador probabilístico ciego de texto**:
* Si le pides: `"Explícame la fotosíntesis"`, el modelo no sabe que le estás haciendo una pregunta a él. Podría continuar con: `"en menos de 50 palabras para un examen de primaria. Pregunta 2: Explica la mitosis..."`.
* Si le pides: `"Escribe un poema sobre el mar"`, podría continuar con: `"dijo el profesor a los alumnos del taller de literatura"`.

### InstructGPT y el Gran Descubrimiento de OpenAI (Ouyang et al., 2022)
En enero de 2022, OpenAI publicó *"Training language models to follow instructions with human feedback"* (InstructGPT):
* Descubrieron que los evaluadores humanos **preferían sistemáticamente las respuestas de un modelo de 1.3B parámetros ajustado con instrucciones frente a un modelo monstruoso de 175B base sin ajustar**.
* Nació la fase de **Supervised Fine-Tuning (SFT)**:
  1. Contrataron anotadores humanos para escribir miles de pares (Instrucción, Respuesta Modelo).
  2. Ajustaron los pesos del modelo pre-entrenado para obedecer y responder de forma directa y útil.
  3. Este fue el verdadero nacimiento del comportamiento de **ChatGPT**.

### La Estandarización de Plantillas de Chat (ChatML)
Para que un modelo distinga los roles en una conversación multi-turno, se crearon formatos estructurados mediante **tokens especiales delimitadores** (ChatML, LLaMA-2 Chat, ChatGLM):
```
<|im_start|>system
Eres un asistente conciso y formal.<|im_end|>
<|im_start|>user
¿Cuál es la distancia a la Luna?<|im_end|>
<|im_start|>assistant
Aproximadamente 384,400 kilómetros.<|im_end|>
```

---

## 2. 🧠 Intuición Geométrica y Mecánica (Mentalidad de Algoritmista)

### El Dilema del Gradiente en SFT
Si concatenamos el Prompt del usuario y la Respuesta del asistente en una sola secuencia de tokens:
$$\text{Secuencia} = [x_1, x_2, \dots, x_P, \; y_1, y_2, \dots, y_R]$$

¿Deberíamos calcular el error de predicción sobre los tokens del prompt ($x_1, \dots, x_P$)?
* **¡Rotundamente NO!** Las preguntas del usuario provienen de una distribución arbitraria y caótica. Si castigamos al modelo por no predecir exactamente las palabras con las que el usuario formula su consulta, desperdiciamos capacidad del gradiente y degradamos su rendimiento.
* Solo queremos optimizar la probabilidad condicional de la **respuesta dada la instrucción**:
  $$\max_\theta \sum_{t=1}^R \log P_\theta(y_t \mid x_1, \dots, x_P, y_1, \dots, y_{t-1})$$

### El Truco del Enmascaramiento con `ignore_index = -100`
En la función de pérdida estándar `nn.CrossEntropyLoss` de PyTorch, existe un parámetro salvador: `ignore_index = -100`.
Cualquier token de etiqueta (*target*) cuyo valor sea `-100` se ignora por completo:
* No contribuye al valor numérico de la pérdida.
* **Su gradiente retropropagado es exactamente 0**.

```
Tokens Input:   [<system>, Eres, un, bot, <user>, Hola, <assistant>, ¡Saludos!, <eos>]
Tokens Target:  [ -100,    -100, -100, -100, -100,  -100,   -100,     ¡Saludos!, <eos>]
                   └── Gradiente = 0 (Ignorado) ──┘       └── Gradiente Activo (SFT) ──┘
```

---

## 3. 🛠️ Implementación "From Scratch" (Primeros Principios)

Implementemos un formateador ChatML, un generador de máscaras de etiquetas con `-100` y un pipeline de entrenamiento SFT.

In [ ]:
# Definición de un vocabulario mínimo con tokens especiales
SPECIAL_TOKENS = {
    "<pad>": 0,
    "<|im_start|>": 1,
    "<|im_end|>": 2,
    "system": 3,
    "user": 4,
    "assistant": 5,
}

class SimpleChatTokenizer:
    """
    Tokenizador básico con soporte para tokens especiales y plantillas de chat.
    """
    def __init__(self):
        self.vocab = dict(SPECIAL_TOKENS)
        self.inverse_vocab = {v: k for k, v in self.vocab.items()}
        
    def add_words(self, words: List[str]):
        for w in words:
            if w not in self.vocab:
                idx = len(self.vocab)
                self.vocab[w] = idx
                self.inverse_vocab[idx] = w

    def encode(self, text: str) -> List[int]:
        # Tokenizado elemental por palabras conservando tokens especiales
        for sp in SPECIAL_TOKENS:
            text = text.replace(sp, f" {sp} ")
        tokens = text.strip().split()
        return [self.vocab.get(t, self.vocab["<pad>"]) for t in tokens]

    def decode(self, ids: List[int]) -> str:
        return " ".join([self.inverse_vocab.get(i, "<?>") for i in ids if i != 0])

    def apply_chat_template(self, messages: List[Dict[str, str]]) -> str:
        """
        Formatea una conversación en formato estándar ChatML.
        """
        formatted = ""
        for msg in messages:
            role = msg["role"]
            content = msg["content"]
            formatted += f"<|im_start|> {role} {content} <|im_end|> "
        return formatted.strip()

tokenizer = SimpleChatTokenizer()
print("✅ SimpleChatTokenizer compilado exitosamente")

### Algoritmo de Enmascaramiento de Etiquetas para SFT
Diseñamos la función clave que analiza la secuencia tokenizada y enmascara todo el contenido que no provenga de los turnos del `assistant`:

In [ ]:
def create_sft_input_and_labels(token_ids: List[int], tokenizer: SimpleChatTokenizer) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Crea input_ids y labels (con -100 en los prompts de system y user).
    """
    im_start_id = tokenizer.vocab["<|im_start|>"]
    im_end_id = tokenizer.vocab["<|im_end|>"]
    assistant_id = tokenizer.vocab["assistant"]
    
    # En modelado causal autorregresivo:
    # input_ids: tokens[0 : N-1]
    # labels:    tokens[1 : N]
    input_ids = token_ids[:-1]
    target_ids = token_ids[1:]
    
    labels = [-100] * len(target_ids)
    
    # Localizar los bloques del asistente y desenmascarar únicamente sus respuestas
    i = 0
    while i < len(input_ids):
        if input_ids[i] == im_start_id and i + 1 < len(input_ids) and input_ids[i+1] == assistant_id:
            # Estamos dentro de la respuesta del asistente
            # Los tokens a predecir comienzan después de '<|im_start|> assistant'
            start_idx = i + 1  # índice en input_ids
            # Buscar '<|im_end|>'
            end_idx = start_idx
            while end_idx < len(input_ids) and input_ids[end_idx] != im_end_id:
                end_idx += 1
            
            # Desenmascarar las etiquetas en el rango correspondiente
            for k in range(start_idx, min(end_idx + 1, len(labels))):
                labels[k] = target_ids[k]
            
            i = end_idx + 1
        else:
            i += 1
            
    return torch.tensor(input_ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

# Ejemplo de prueba con una conversación
sample_convo = [
    {"role": "system", "content": "Eres un asistente experto en algoritmos"},
    {"role": "user", "content": "Cual es la complejidad de Quicksort"},
    {"role": "assistant", "content": "O(N log N) en promedio"}
]

# Extraer palabras y agregarlas al vocabulario
text_convo = tokenizer.apply_chat_template(sample_convo)
tokenizer.add_words(text_convo.replace("<|im_start|>", "").replace("<|im_end|>", "").split())

encoded_ids = tokenizer.encode(text_convo)
inputs, targets = create_sft_input_and_labels(encoded_ids, tokenizer)

print("Texto formateado:", text_convo)
print("\n--- Alineamiento de Tokens e Índices de Pérdida ---")
for inp, tgt in zip(inputs.tolist(), targets.tolist()):
    word_inp = tokenizer.inverse_vocab.get(inp, "<?>")
    lbl_str = str(tgt) if tgt != -100 else "-100 (IGNORADO)"
    print(f"Input: {word_inp:<15} -> Target: {lbl_str}")

### Demostración Matemática: Pérdida Enmascarada en PyTorch
Comprobemos que `nn.CrossEntropyLoss(ignore_index=-100)` ignora rigurosamente los tokens de prompt y concentra el gradiente exclusivamente en la respuesta del asistente:

In [ ]:
vocab_size = len(tokenizer.vocab)
seq_len = len(inputs)
# Logits sintéticos simulados de una red neuronal
logits = torch.randn(seq_len, vocab_size, requires_grad=True)

criterion = nn.CrossEntropyLoss(ignore_index=-100)
loss = criterion(logits, targets)
loss.backward()

# Inspeccionar la norma del gradiente para cada posición
grad_norms = logits.grad.norm(dim=-1).numpy()

print(f"Pérdida total calculada: {loss.item():.4f}")
print("\n--- Magnitud de Gradiente por Token ---")
for idx, (inp, tgt, g) in enumerate(zip(inputs.tolist(), targets.tolist(), grad_norms)):
    w = tokenizer.inverse_vocab.get(inp, "<?>")
    status = "ACTIVO" if tgt != -100 else "IGNORADO"
    print(f"Posición {idx:02d} [{w:<12}] | Target: {tgt:>4} | Gradiente: {g:.4f} ({status})")
    if tgt == -100:
        assert np.isclose(g, 0.0), f"¡Error! El gradiente en token ignorado debe ser 0, pero dio {g}"

print("\n✅ Verificación exitosa: Todos los tokens de prompt tienen gradiente estrictamente CERO")

---

## 4. ⚡ Transición a PyTorch Moderno

En la industria actual, no construimos los bucles de enmascaramiento a mano. La librería estándar es **`TRL` (Transformer Reinforcement Learning)** de Hugging Face mediante la clase `DataCollatorForCompletionOnlyLM`:

```python
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

# Indicamos el delimitador exacto de inicio de respuesta
response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer
)

# El SFTTrainer aplica la máscara de pérdida automáticamente en segundo plano
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    data_collator=collator,
    max_seq_length=2048
)
```

---

## 5. 🎯 Retos & Experimentos ("Tinker Time")

### Reto 1: Cálculo Manual de Cross-Entropy Enmascarada en NumPy
Calcula manualmente la pérdida Cross-Entropy en NumPy filtrando las posiciones válidas ($y \neq -100$) y comprueba que coincide con PyTorch hasta 5 decimales:

In [ ]:
with torch.no_grad():
    logits_np = logits.numpy()
    targets_np = targets.numpy()
    
    # 1. Softmax numérico estable en NumPy
    shift_logits = logits_np - np.max(logits_np, axis=-1, keepdims=True)
    probs = np.exp(shift_logits) / np.sum(np.exp(shift_logits), axis=-1, keepdims=True)
    
    # 2. Filtrar únicamente las posiciones válidas (targets != -100)
    valid_mask = (targets_np != -100)
    valid_targets = targets_np[valid_mask]
    valid_probs = probs[valid_mask]
    
    # 3. Log-loss: -log(p[target])
    correct_probs = valid_probs[np.arange(len(valid_targets)), valid_targets]
    manual_loss = -np.mean(np.log(correct_probs))
    
    pytorch_loss = loss.item()
    print(f"Pérdida manual en NumPy: {manual_loss:.5f}")
    print(f"Pérdida en PyTorch:      {pytorch_loss:.5f}")
    assert np.isclose(manual_loss, pytorch_loss, atol=1e-4)
    print("✅ La formulación analítica de la pérdida SFT coincide con exactitud numérica")

### Reto 2 (Para resolver): Implementar Sequence Packing
Cuando entrenamos modelos en batches, usar padding rellenando con `<pad>` desperdicia memoria GPU. La técnica de **Sequence Packing** concatena múltiples conversaciones cortas en una única secuencia 1D hasta alcanzar un límite fijo `max_seq_len`, devolviendo también las máscaras correspondientes.

Implementa a continuación la función `pack_sequences(sequences, max_seq_len)`:

In [ ]:
# TU CÓDIGO DEL RETO 2 AQUÍ
def pack_sequences(conversations: List[List[int]], max_seq_len: int = 16) -> List[List[int]]:
    """
    Empaqueta múltiples secuencias de tokens en bloques compactos de longitud fija.
    """
    # Tu implementación aquí
    pass

---

## 6. 📚 Referencias Fundamentales & Lecturas Recomendadas

### 📄 Papers Seminales
1. **Ouyang, L., et al. (2022):** *"Training language models to follow instructions with human feedback"* (InstructGPT), NeurIPS 2022. [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)
   * *¿Qué leer?* Sección 3 ("Methodology"): la estructura del pipeline de SFT previo a la fase de alineamiento.
2. **Taori, R., et al. (2023):** *"Alpaca: A Strong, Replicable Instruction-Following Model"*, Stanford CRFM. [Blog Post](https://crfm.stanford.edu/2023/03/13/alpaca.html)
   * *¿Qué leer?* La generación sintética de 52,000 pares de instrucciones usando text-davinci-003 para democratizar el SFT de código abierto.
3. **Zheng, L., et al. (2023):** *"Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena"*, NeurIPS 2023. [arXiv:2306.05685](https://arxiv.org/abs/2306.05685)
   * *¿Qué leer?* Métodos de evaluación estandarizados para medir la calidad conversacional de modelos post-entrenados.